In [3]:
import numpy as np
from collections import Counter
from ase import Atoms
from ase.io import read, write

from chempiler.perception import build_molecules

TRAJFILE_ion    = '../data/MACE-POLAR-1-M/39_water_OH/charged_singlet/39_water_OH.traj'
TRAJFILE_rad    = '../data/MACE-POLAR-1-M/39_water_OH/neutral_doublet/39_water_OH.traj'


def formula_of(syms, mol):
    c = Counter(syms[i] for i in mol)
    return ''.join(f'{el}{n}' for el, n in sorted(c.items()))


def centered_last_frame(path, out_path, bond_scale=0.8):
    """
    Read the last frame, find the single non-water species (the OH-/OH*
    centre, possibly merged with a neighbour into e.g. H3O2 at that instant),
    and wrap the whole box around its centre of mass using MIC - molecule by
    molecule, so every water (and the special species) is translated as a
    rigid unit and stays intact. Written with no cell/lattice: a finite,
    non-periodic cluster with the special species intact at the centre.
    """
    atoms = read(path, index=-1)
    syms = atoms.get_chemical_symbols()
    pos = atoms.get_positions().copy()
    masses = atoms.get_masses()
    cell = np.asarray(atoms.get_cell())
    inv_cell = np.linalg.inv(cell)

    mols = build_molecules(atoms, mode='molecular', bond_scale=bond_scale)
    special = [mol for mol in mols if formula_of(syms, mol) != 'H2O1']
    assert len(special) == 1, (
        f"expected exactly one non-water species, got "
        f"{[formula_of(syms, m) for m in special]}"
    )
    special = special[0]
    formula = formula_of(syms, special)

    # 1. Make every molecule internally whole: fix any atom that got wrapped
    #    to the wrong periodic image relative to the rest of its own molecule.
    for mol in mols:
        if len(mol) < 2:
            continue
        ref = pos[mol[0]]
        for idx in mol[1:]:
            diff = pos[idx] - ref
            frac = diff @ inv_cell
            frac -= np.round(frac)
            pos[idx] = ref + frac @ cell

    # 2. Mass-weighted COM of the special species, now that it's whole.
    anchor = np.average(pos[special], axis=0, weights=masses[special])

    # 3. Shift each molecule as a rigid unit (same translation for every atom
    #    in it) so its own COM lands in the periodic image nearest the anchor.
    #    This is what keeps molecules intact -- no atom is ever wrapped alone.
    out_pos = pos.copy()
    for mol in mols:
        mol = np.asarray(mol)
        mol_com = np.average(pos[mol], axis=0, weights=masses[mol])
        frac = (mol_com - anchor) @ inv_cell
        shift = -np.round(frac) @ cell
        out_pos[mol] = pos[mol] + shift

    out_pos -= anchor

    centered = Atoms(symbols=syms, positions=out_pos, pbc=False)
    write(out_path, centered)

    max_bond = max(
        np.linalg.norm(out_pos[i] - out_pos[j])
        for mol in mols for i in mol for j in mol if i < j
    )
    print(
        f"{out_path}: centred on {formula} COM, {len(centered)} atoms, "
        f"max intramolecular distance {max_bond:.3f} A, no cell/lattice"
    )
    return centered


ion_snapshot = centered_last_frame(TRAJFILE_ion, 'ORCA/39_water_OH_charged_singlet_last.xyz')
rad_snapshot = centered_last_frame(TRAJFILE_rad, 'ORCA/39_water_OH_neutral_doublet_last.xyz')

ORCA/39_water_OH_charged_singlet_last.xyz: centred on H3O2 COM, 119 atoms, max intramolecular distance 2.916 A, no cell/lattice
ORCA/39_water_OH_neutral_doublet_last.xyz: centred on H1O1 COM, 119 atoms, max intramolecular distance 1.657 A, no cell/lattice


In [4]:
from ase.io import read, write
from ase.optimize import BFGS
from mace.calculators import mace_polar

DEVICE = 'cpu'  # switch to 'cuda' if running on a GPU machine

ion_snapshot = read('ORCA/39_water_OH_charged_singlet_1st.xyz')
rad_snapshot = read('ORCA/39_water_OH_neutral_doublet_1st.xyz')


def optimize_cluster(atoms, charge, spin, out_path, fmax=0.05, steps=500, device=DEVICE):
    """
    Relax a finite (pbc=False) cluster with the MACE-POLAR-1-M potential,
    same charge/spin convention as data/SCRIPTS/md.py.
    """
    atoms = atoms.copy()
    atoms.info['charge'] = charge
    atoms.info['spin'] = spin
    atoms.info['external_field'] = [0.0, 0.0, 0.0]
    atoms.calc = mace_polar(model="polar-1-m", default_dtype="float32", device=device)

    dyn = BFGS(
        atoms,
        trajectory=out_path.replace('.xyz', '_opt.traj'),
        logfile=out_path.replace('.xyz', '_opt.log'),
    )
    dyn.run(fmax=fmax, steps=steps)

    write(out_path, atoms)
    print(
        f"{out_path}: relaxed in {dyn.nsteps} steps "
        f"(fmax<{fmax} eV/A), Epot={atoms.get_potential_energy():.4f} eV"
    )
    return atoms


ion_opt = optimize_cluster(
    ion_snapshot, charge=-1, spin=1,
    out_path='ORCA/39_water_OH_charged_singlet_opt.xyz',
)
rad_opt = optimize_cluster(
    rad_snapshot, charge=0, spin=2,
    out_path='ORCA/39_water_OH_neutral_doublet_opt.xyz',
)

/home/viktor/.local/lib/python3.10/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


Using MACE-Polar model for MACECalculator with /home/viktor/.cache/mace/MACEPOLAR1Mmodel


/home/viktor/.local/lib/python3.10/site-packages/mace/calculators/mace.py:207: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


ORCA/39_water_OH_charged_singlet_opt.xyz: relaxed in 222 steps (fmax<0.05 eV/A), Epot=-18707.5938 eV
Using MACE-Polar model for MACECalculator with /home/viktor/.cache/mace/MACEPOLAR1Mmodel


/home/viktor/.local/lib/python3.10/site-packages/mace/calculators/mace.py:207: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


ORCA/39_water_OH_neutral_doublet_opt.xyz: relaxed in 157 steps (fmax<0.05 eV/A), Epot=-18702.8652 eV
